# Approximate BR calibration against a neural reference policy

This notebook calibrates neural approximate best response against the final learned-average policy from a Deep CFR reference run on `r4_s4_h2_hp2pt_ss`.

At every checkpoint it measures the learned responder in two ways:

1. **Exact fixed-response value:** compile the greedy Q responder to a dense policy and evaluate it exactly against the dense reference policy.
2. **High-volume Monte Carlo value:** play many held-out games using a fixed monitoring stream.

The exact optimal BR is calculated once. This lets us test whether early exact and Monte Carlo approximate-BR curves predict the true exploitability.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import curve_fit

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy
from liars_poker.policies.neural import NeuralPolicy, compile_neural_to_dense
from liars_poker.policies.tabular_dense import DenseTabularPolicy
from liars_poker.algo.br_exact_dense_to_dense import (
    BestResponseComputerDense,
    best_response_dense,
)
from liars_poker.algo.br_neural import NeuralBRTrainer
from liars_poker.eval.approx_br import wilson_lower_bound

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Load the rerun reference policy

Set `REFERENCE_RUN_DIR` explicitly if desired. If it is `None`, the newest completed reference directory containing a serialized policy is selected.

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

REFERENCE_RUN_DIR = None
reference_root = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs'
if REFERENCE_RUN_DIR is None:
    candidates = sorted(
        (
            path for path in reference_root.glob(f'{spec.to_short_str()}___*')
            if (path / 'policy' / 'metadata.json').exists()
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            'No completed reference policy found. Rerun deep_cfr_gpu_reference_run.ipynb first.'
        )
    REFERENCE_RUN_DIR = candidates[0]
else:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)

target_policy, target_spec = load_policy(str(REFERENCE_RUN_DIR / 'policy'))
assert target_spec == spec
assert isinstance(target_policy, NeuralPolicy)

# The serializer loads neural policies on CPU. Move the frozen target to the
# selected device for dense compilation and GPU rollout evaluation.
target_policy.model_p1.to(device)
target_policy.model_p2.to(device)
target_policy.device = torch.device(device)
target_policy.eval()

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT_DIR = (
    REPO_ROOT / 'artifacts' / 'neural_br_reference_calibration'
    / f'{spec.to_short_str()}___{run_id}'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_JSONL = OUTPUT_DIR / 'results.jsonl'

print('Reference run:', REFERENCE_RUN_DIR)
print('Output directory:', OUTPUT_DIR)
print(target_policy)

## Exact target ceiling

The target neural policy is compiled to dense form once. The trusted dense best-response solver then gives the true role-specific ceilings and exact exploitability.

In [ ]:
start = time.perf_counter()
target_dense = compile_neural_to_dense(target_policy, batch_size=65_536)
target_compile_s = time.perf_counter() - start

start = time.perf_counter()
exact_br_policy, exact_meta = best_response_dense(
    spec,
    target_dense,
    debug=True,
    store_state_values=False,
)
exact_br_s = time.perf_counter() - start
exact_first, exact_second = exact_meta['computer'].exploitability()
exact_exploitability = exact_first + exact_second - 1.0

pd.Series({
    'exact BR win rate as first': exact_first,
    'exact BR win rate as second': exact_second,
    'exact exploitability': exact_exploitability,
    'target dense compile s': target_compile_s,
    'exact BR s': exact_br_s,
})

## Efficient exact value of a fixed learned responder

`BestResponseComputerDense` already contains the correct blocker-aware recursion. The class below changes only responder nodes: instead of maximizing over actions, it follows the supplied responder policy. Opponent nodes and terminal resolution remain exactly those of the trusted solver.

In [ ]:
class FixedResponseComputerDense(BestResponseComputerDense):
    def __init__(self, spec, opponent, responder):
        super().__init__(spec, opponent, store_state_values=False)
        if responder.spec != spec or responder.S.shape != opponent.S.shape:
            raise ValueError('Responder and opponent dense spaces do not match.')
        self.responder = responder

    def _solve_node(self, hid, opp_reach, *, to_play, my_id):
        if to_play != my_id:
            return super()._solve_node(
                hid, opp_reach, to_play=to_play, my_id=my_id
            )

        actions = self.opponent.legal_actions[hid]
        if not actions:
            zeros = np.zeros(self.n_hands, dtype=float)
            return zeros, zeros

        total_wm = np.zeros(self.n_hands, dtype=float)
        total_m = np.zeros(self.n_hands, dtype=float)
        next_to_play = 1 - to_play
        strategy = self.responder.S[hid]

        for action in actions:
            col = 0 if action == -1 else action + 1
            if action == -1:
                wm, mass = self._solve_terminal(
                    hid, opp_reach, caller=to_play, my_id=my_id
                )
            else:
                wm, mass = self._solve_node(
                    hid | (1 << action),
                    opp_reach,
                    to_play=next_to_play,
                    my_id=my_id,
                )
            probabilities = strategy[:, col].astype(float, copy=False)
            total_wm += probabilities * wm
            total_m += probabilities * mass

        return total_wm, total_m


def compile_q_to_dense(policy, batch_size=65_536):
    dense = DenseTabularPolicy(policy.spec)
    dense.S.fill(0.0)
    n_hands = len(dense.hands)
    histories_per_batch = max(1, int(batch_size) // n_hands)
    rank_dim = policy.spec.ranks
    input_dim = policy.encoder.input_dim
    action_dim = policy.encoder.action_dim
    claim_bits = np.arange(policy.encoder.k, dtype=np.int64)
    hand_features = policy.encoder.encode_hands(dense.hands, ())[:, :rank_dim]

    with torch.inference_mode():
        for pid in (0, 1):
            actor_hids = np.flatnonzero((dense.popcount & 1) == pid)
            model = policy._model(pid)
            for start in range(0, len(actor_hids), histories_per_batch):
                hids = actor_hids[start:start + histories_per_batch]
                history_bits = (
                    (hids[:, None].astype(np.int64) >> claim_bits[None, :]) & 1
                ).astype(np.float32)
                features = np.empty(
                    (len(hids), n_hands, input_dim), dtype=np.float32
                )
                features[:, :, :rank_dim] = hand_features[None, :, :]
                features[:, :, rank_dim:] = history_bits[:, None, :]
                x = torch.from_numpy(features.reshape(-1, input_dim)).to(policy.device)
                q = model(x).reshape(len(hids), n_hands, action_dim)
                legal = torch.from_numpy(dense.legal_mask[hids]).to(policy.device)
                best_cols = q.masked_fill(~legal[:, None, :], -torch.inf).argmax(dim=2)
                best_cols = best_cols.cpu().numpy()
                block = np.zeros(
                    (len(hids), n_hands, action_dim), dtype=np.float32
                )
                block[
                    np.arange(len(hids))[:, None],
                    np.arange(n_hands)[None, :],
                    best_cols,
                ] = 1.0
                dense.S[hids] = block

    dense.recompute_likelihoods()
    return dense


def exact_fixed_response_values(responder):
    start = time.perf_counter()
    computer = FixedResponseComputerDense(spec, target_dense, responder)
    computer.solve()
    p_first, p_second = computer.exploitability()
    elapsed = time.perf_counter() - start
    del computer
    return float(p_first), float(p_second), elapsed

In [ ]:
# One-time implementation check: following the exact BR policy must reproduce
# the exact BR values. This checks both seats and all blocker-aware weighting.
check_first, check_second, check_s = exact_fixed_response_values(exact_br_policy)
print('exact oracle:', exact_first, exact_second)
print('fixed evaluator:', check_first, check_second, f'({check_s:.2f}s)')
assert np.isclose(check_first, exact_first, atol=1e-6)
assert np.isclose(check_second, exact_second, atol=1e-6)

del exact_br_policy, exact_meta
gc.collect()

## Monitoring functions

Monte Carlo evaluation uses the same role-specific random streams at every checkpoint. The trainer's original generator state is restored afterward, so diagnostics cannot alter future training.

In [ ]:
def exact_current_response(trainer):
    start = time.perf_counter()
    responder = compile_q_to_dense(trainer.policy())
    compile_s = time.perf_counter() - start
    p_first, p_second, solve_s = exact_fixed_response_values(responder)
    del responder
    gc.collect()
    return {
        'exact_p_first': p_first,
        'exact_p_second': p_second,
        'exact_discovered': p_first + p_second - 1.0,
        'responder_compile_s': compile_s,
        'exact_fixed_eval_s': solve_s,
    }


def monte_carlo_current_response(
    trainer,
    episodes_per_role,
    *,
    evaluation_seed,
    rollout_batch_size=8192,
    z=1.96,
):
    start = time.perf_counter()
    training_generator_state = trainer.generator.get_state()
    role_results = []
    try:
        for role in (0, 1):
            trainer.generator.manual_seed(int(evaluation_seed + role))
            role_results.append(
                trainer.evaluate_role(
                    role,
                    episodes_per_role,
                    rollout_batch_size=rollout_batch_size,
                )
            )
    finally:
        trainer.generator.set_state(training_generator_state)

    p_first = float(role_results[0]['win_rate'])
    p_second = float(role_results[1]['win_rate'])
    first_lcb = wilson_lower_bound(
        int(role_results[0]['wins']), int(role_results[0]['episodes']), z=z
    )
    second_lcb = wilson_lower_bound(
        int(role_results[1]['wins']), int(role_results[1]['episodes']), z=z
    )
    return {
        'mc_p_first': p_first,
        'mc_p_second': p_second,
        'mc_discovered': p_first + p_second - 1.0,
        'mc_p_first_lcb': first_lcb,
        'mc_p_second_lcb': second_lcb,
        'mc_conservative_lower_bound': first_lcb + second_lcb - 1.0,
        'mc_episodes_per_role': int(episodes_per_role),
        'mc_eval_s': time.perf_counter() - start,
    }


def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'Not JSON serializable: {type(value)!r}')


def append_jsonl(path, record):
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, default=json_default) + '\n')

## Three-seed calibration run

No responder checkpoints are written. Only small JSONL/CSV diagnostics are retained, avoiding the replay-buffer disk cost.

The default experiment uses fifteen measured training minutes per seed and one million monitoring games per role at every checkpoint.

In [ ]:
SEEDS = (7, 17, 27)
TRAINING_BUDGETS_S = (0, 15, 30, 60, 120, 240, 480, 900)
MONITOR_EPISODES_PER_ROLE = 1_000_000
FINAL_FRESH_EPISODES_PER_ROLE = 2_000_000
MONITOR_EVALUATION_SEED = 91_000

TRAINER_KWARGS = dict(
    hidden_sizes=(512, 512),
    device=device,
    expansion='sampled',
    replay_capacity=1_000_000,
    batch_size=4096,
    learning_rate=1e-3,
    train_steps=100,
    warmup_transitions=20_000,
    target_update_every=500,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_decisions=500_000,
)
EPISODES_PER_ROLE = 4096
ROLLOUT_BATCH_SIZE = 4096

manifest = {
    'reference_run': str(REFERENCE_RUN_DIR),
    'spec': spec.to_json(),
    'seeds': list(SEEDS),
    'training_budgets_s': list(TRAINING_BUDGETS_S),
    'monitor_episodes_per_role': MONITOR_EPISODES_PER_ROLE,
    'trainer_kwargs': {**TRAINER_KWARGS, 'device': str(device)},
    'exact_first': exact_first,
    'exact_second': exact_second,
    'exact_exploitability': exact_exploitability,
}
(OUTPUT_DIR / 'manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)

In [ ]:
all_rows = []

for seed in SEEDS:
    print(f'\n=== seed={seed} ===')
    trainer = NeuralBRTrainer(spec, target_policy, seed=seed, **TRAINER_KWARGS)
    measured_training_s = 0.0
    next_budget_idx = 0
    best_exact_first = -np.inf
    best_exact_second = -np.inf
    best_mc_first = -np.inf
    best_mc_second = -np.inf
    last_training_record = None

    while next_budget_idx < len(TRAINING_BUDGETS_S):
        requested_budget = TRAINING_BUDGETS_S[next_budget_idx]
        if measured_training_s >= requested_budget:
            exact_result = exact_current_response(trainer)
            mc_result = monte_carlo_current_response(
                trainer,
                MONITOR_EPISODES_PER_ROLE,
                evaluation_seed=MONITOR_EVALUATION_SEED,
            )

            best_exact_first = max(best_exact_first, exact_result['exact_p_first'])
            best_exact_second = max(best_exact_second, exact_result['exact_p_second'])
            best_mc_first = max(best_mc_first, mc_result['mc_p_first'])
            best_mc_second = max(best_mc_second, mc_result['mc_p_second'])

            row = {
                'seed': seed,
                'requested_budget_s': requested_budget,
                'measured_training_s': measured_training_s,
                'iteration': trainer.iteration,
                **exact_result,
                **mc_result,
                'best_exact_p_first': best_exact_first,
                'best_exact_p_second': best_exact_second,
                'best_exact_discovered': best_exact_first + best_exact_second - 1.0,
                'best_mc_p_first': best_mc_first,
                'best_mc_p_second': best_mc_second,
                'best_mc_discovered': best_mc_first + best_mc_second - 1.0,
            }
            row['exact_oracle_gap'] = max(
                0.0, exact_exploitability - row['best_exact_discovered']
            )
            row['mc_oracle_gap'] = max(
                0.0, exact_exploitability - row['best_mc_discovered']
            )
            row['mc_minus_exact_current'] = (
                row['mc_discovered'] - row['exact_discovered']
            )
            if last_training_record is not None:
                for role in (0, 1):
                    role_record = last_training_record['roles'][role]
                    row[f'role_{role + 1}_loss'] = role_record['loss']
                    row[f'role_{role + 1}_replay_seen'] = role_record['replay_seen']
                    row[f'role_{role + 1}_epsilon'] = role_record['epsilon']

            all_rows.append(row)
            append_jsonl(RESULTS_JSONL, row)
            print(
                f"train={measured_training_s:7.1f}s iter={trainer.iteration:4d} "
                f"exact={row['exact_discovered']:.5f} "
                f"best_exact={row['best_exact_discovered']:.5f} "
                f"mc={row['mc_discovered']:.5f} "
                f"gap={row['exact_oracle_gap']:.5f}"
            )
            next_budget_idx += 1
            continue

        start = time.perf_counter()
        last_training_record = trainer.run_iteration(
            episodes_per_role=EPISODES_PER_ROLE,
            rollout_batch_size=ROLLOUT_BATCH_SIZE,
        )
        measured_training_s += time.perf_counter() - start

    fresh_mc = monte_carlo_current_response(
        trainer,
        FINAL_FRESH_EPISODES_PER_ROLE,
        evaluation_seed=MONITOR_EVALUATION_SEED + 1_000_000 + seed * 10,
    )
    all_rows[-1].update({f'fresh_{key}': value for key, value in fresh_mc.items()})

    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = pd.DataFrame(all_rows)
results.to_csv(OUTPUT_DIR / 'results.csv', index=False)
results.tail()

## Exact versus Monte Carlo progress

The exact curve diagnoses responder learning. The Monte Carlo curve tests what the same experiment would look like after exact fixed-policy evaluation becomes unavailable.

In [ ]:
summary_rows = []
for seed, frame in results.groupby('seed'):
    final = frame.sort_values('measured_training_s').iloc[-1]
    summary_rows.append({
        'seed': seed,
        'iterations': final['iteration'],
        'final exact current': final['exact_discovered'],
        'final exact best': final['best_exact_discovered'],
        'final exact oracle gap': final['exact_oracle_gap'],
        'fraction exact exploitability recovered': (
            final['best_exact_discovered'] / exact_exploitability
        ),
        'final monitoring MC current': final['mc_discovered'],
        'final fresh MC current': final.get('fresh_mc_discovered', np.nan),
        'monitor MC minus exact': final['mc_minus_exact_current'],
        'mean exact fixed eval s': frame['exact_fixed_eval_s'].mean(),
        'mean MC eval s': frame['mc_eval_s'].mean(),
    })

summary = pd.DataFrame(summary_rows).set_index('seed')
display(summary)
display(summary.agg(['mean', 'std']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for seed, frame in results.groupby('seed'):
    frame = frame.sort_values('measured_training_s')
    positive = frame['measured_training_s'] > 0
    x = frame.loc[positive, 'measured_training_s']
    axes[0].plot(x, frame.loc[positive, 'best_exact_discovered'], marker='o', label=f'seed {seed}')
    axes[1].plot(x, frame.loc[positive, 'best_mc_discovered'], marker='o', label=f'seed {seed}')
    axes[2].loglog(
        x,
        frame.loc[positive, 'exact_oracle_gap'].clip(lower=1e-10),
        marker='o', label=f'seed {seed}',
    )

axes[0].axhline(exact_exploitability, color='black', linestyle='--', label='exact ceiling')
axes[1].axhline(exact_exploitability, color='black', linestyle='--', label='exact ceiling')
axes[0].set_title('Exact best discovered lower bound')
axes[1].set_title('Monte Carlo best discovered estimate')
axes[2].set_title('Exact gap to optimal response')
for ax in axes[:2]:
    ax.set_xscale('log')
for ax in axes:
    ax.set_xlabel('Measured responder-training seconds')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].scatter(
    results['exact_discovered'],
    results['mc_discovered'],
    c=results['measured_training_s'],
    cmap='viridis',
    alpha=0.8,
)
lower = min(results['exact_discovered'].min(), results['mc_discovered'].min())
upper = max(results['exact_discovered'].max(), results['mc_discovered'].max())
axes[0].plot([lower, upper], [lower, upper], color='black', linestyle='--')
axes[0].set_xlabel('Exact current discovered exploitability')
axes[0].set_ylabel('Monte Carlo current estimate')
axes[0].set_title('Monte Carlo calibration')

for seed, frame in results.groupby('seed'):
    axes[1].plot(
        frame['measured_training_s'],
        frame['mc_minus_exact_current'],
        marker='o',
        label=f'seed {seed}',
    )
axes[1].axhline(0.0, color='black', linestyle='--')
axes[1].set_xscale('symlog', linthresh=15)
axes[1].set_xlabel('Measured responder-training seconds')
axes[1].set_ylabel('MC estimate minus exact value')
axes[1].set_title('Monitoring error over training')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()

## Early-curve prediction of exact exploitability

For each cutoff we fit

$$L(t)=E_\infty-c(t+1)^{-\alpha}$$

to the cumulative best exact curve and cumulative best Monte Carlo curve. The fitted asymptote is compared with the known exact exploitability. These are heuristic forecasts, not certified bounds.

In [ ]:
def asymptotic_curve(t, asymptote, scale, alpha):
    return asymptote - scale * np.power(t + 1.0, -alpha)


def fit_asymptote(frame, value_column, cutoff_s):
    fit = frame[
        (frame['measured_training_s'] > 0)
        & (frame['measured_training_s'] <= cutoff_s)
    ].sort_values('measured_training_s')
    if len(fit) < 4:
        raise ValueError('Need at least four positive-time points.')
    t = fit['measured_training_s'].to_numpy(dtype=float)
    y = fit[value_column].to_numpy(dtype=float)
    lower_bound = float(y.max())
    initial_asymptote = min(0.999, lower_bound + max(0.01, np.ptp(y)))
    initial_scale = max(1e-4, initial_asymptote - y[0])
    params, _ = curve_fit(
        asymptotic_curve,
        t,
        y,
        p0=(initial_asymptote, initial_scale, 0.5),
        bounds=((lower_bound, 0.0, 0.01), (1.0, 10.0, 3.0)),
        maxfev=20_000,
    )
    return float(params[0]), float(params[2])


prediction_rows = []
for seed, frame in results.groupby('seed'):
    for cutoff_s in (120, 240, 480, 900):
        for source, column in (
            ('exact checkpoints', 'best_exact_discovered'),
            ('Monte Carlo checkpoints', 'best_mc_discovered'),
        ):
            try:
                prediction, alpha = fit_asymptote(frame, column, cutoff_s)
                prediction_rows.append({
                    'seed': seed,
                    'cutoff_s': cutoff_s,
                    'source': source,
                    'predicted exploitability': prediction,
                    'prediction error': prediction - exact_exploitability,
                    'fitted alpha': alpha,
                })
            except (RuntimeError, ValueError) as exc:
                prediction_rows.append({
                    'seed': seed,
                    'cutoff_s': cutoff_s,
                    'source': source,
                    'fit error': str(exc),
                })

predictions = pd.DataFrame(prediction_rows)
display(predictions.set_index(['source', 'cutoff_s', 'seed']))
display(
    predictions.groupby(['source', 'cutoff_s'])[
        ['predicted exploitability', 'prediction error', 'fitted alpha']
    ].agg(['mean', 'std'])
)

fig, ax = plt.subplots(figsize=(8, 5))
for source, group in predictions.dropna(subset=['predicted exploitability']).groupby('source'):
    aggregate = group.groupby('cutoff_s')['predicted exploitability'].agg(['mean', 'std'])
    ax.errorbar(
        aggregate.index / 60,
        aggregate['mean'],
        yerr=aggregate['std'].fillna(0.0),
        marker='o',
        capsize=4,
        label=source,
    )
ax.axhline(exact_exploitability, color='black', linestyle='--', label='known exact')
ax.set_xlabel('Training data available to fit (minutes)')
ax.set_ylabel('Fitted asymptotic exploitability')
ax.set_title('How early can approximate BR predict exact exploitability?')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()